# Building Credit Risk Assessment Model Responsibly

In this notebook, we use a dataset prepared from the UCI South German Credit.

> "South German Credit," UCI Machine Learning Repository, 2020. [Online]. Available: https://doi.org/10.24432/C5QG88.

## Task 4: Rebuild a new credit risk assessment model

In this task, we recreate and deploy a new trained model.

### Task validation: 
Submit the auc score and the endpoint name in the validation input following this format `auc_score|endpoint_name`. 

For example: `0.12345|responsible-endpoint-123`.

In [ ]:
import boto3
import datetime as dt
import pandas as pd
import sagemaker

from sagemaker import get_execution_role
from sagemaker import clarify

from sagemaker.sklearn import SKLearnModel
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput, ProcessingJob
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.xgboost import XGBoostPredictor
from sagemaker.pipeline import PipelineModel
from sagemaker.predictor import Predictor
from sklearn.metrics import roc_auc_score

In [ ]:
# Important constant variables
BUCKET = '<LabBucketName>' #Replace with LabBucketName in the challenge output properties

In [ ]:
# Important constant variables
# DO NOT CHANGE the value of these constants

INSTANCE_TYPE = 'ml.m5.xlarge' # Use this instance type for all processing, and training jobs, as well as the real-time endpoints.

S3_DATA_PREFIX = 'data'
TRAIN_FILE = 'data/train.csv'
VAL_FILE = 'data/validation.csv'
TEST_FILE = 'data/test.csv'

train_s3_path = f"s3://{BUCKET}/{TRAIN_FILE}"
val_s3_path = f"s3://{BUCKET}/{VAL_FILE}"
test_s3_path = f"s3://{BUCKET}/{TEST_FILE}"

DATASET_COLUMNS = [
    "credit_risk",
    "status",
    "duration",
    "credit_history",
    "purpose",
    "amount",
    "savings",
    "employment_duration",
    "installment_rate",
    "personal_status_sex",
    "other_debtors",
    "present_residence",
    "property",
    "age",
    "other_installment_plans",
    "housing",
    "number_credits",
    "job",
    "people_liable",
    "telephone",
    "foreign_worker",
]
LABEL_COLUMN = 'credit_risk'

In [ ]:
print(train_s3_path)
print(val_s3_path)
print(test_s3_path)

In [ ]:
sess = boto3.Session()
region = sess.region_name
sagemaker_session = sagemaker.Session(
    boto_session=sess,
    default_bucket=BUCKET,  # Use the jam-provided bucket instead of a new default bucket
)
role = get_execution_role()

## Data Preprocessing

Beside performing one-hot encoding on several categorical columns, we also want to drop sensitive columns `personal_status_sex` and `age`. We use `SKLearnProcessor` container to run `script/preprocessor2.py` which will perform the transformation. You don't need to edit the python script file.

In [ ]:
sklearn_processor = SKLearnProcessor(
    sagemaker_session=sagemaker_session,
    role=role,
    framework_version='1.2-1',
    instance_type=INSTANCE_TYPE,
    instance_count=1,
)

PROC_INPUT_PATH = '/opt/ml/processing/input'
base_proc_job_name = f'responsible-ai-jam-processing-{dt.datetime.now().strftime("%Y-%m-%d-%H-%M-%S")}'
sklearn_processor.run(
    inputs=[
        ProcessingInput(source=f's3://{BUCKET}/{S3_DATA_PREFIX}/', destination=PROC_INPUT_PATH),
    ],
    outputs=[
        ProcessingOutput(output_name='processing_model', source='/opt/ml/processing/model'),
        ProcessingOutput(output_name='train', source='/opt/ml/processing/train'),
        ProcessingOutput(output_name='validation', source='/opt/ml/processing/validation'),
    ],
    code='scripts/preprocessor2.py',
    job_name=base_proc_job_name,

)

In [ ]:
proc_job_info = ProcessingJob.from_processing_name(
    sagemaker_session,
    base_proc_job_name,
).describe()

output_config = proc_job_info['ProcessingOutputConfig']
for output in output_config['Outputs']:
    if output['OutputName'] == 'train':
        preprocessed_train_data = output['S3Output']['S3Uri']
    if output['OutputName'] == 'validation':
        preprocessed_val_data = output['S3Output']['S3Uri']
    if output['OutputName'] == 'processing_model':
        processing_model_s3_path = output['S3Output']['S3Uri']

In [ ]:
preprocessed_train_data

In [ ]:
preprocessed_val_data

In [ ]:
processing_model_s3_path

## XGBoost training

We train a machine learning model using Amazon SageMaker XGBoost built-in algorithm.

In [ ]:
image_uri = sagemaker.image_uris.retrieve(
    framework='xgboost',
    region=region,
    version='1.7-1'
)

training_job_name = f'responsible-training-{dt.datetime.now().strftime("%Y-%m-%d-%H-%M-%S")}'
base_model_s3_path = f's3://{BUCKET}/model/base/'
xgb_train = Estimator(
    sagemaker_session=sagemaker_session,
    role=role,
    image_uri=image_uri,
    instance_type=INSTANCE_TYPE,
    instance_count=1,
    output_path=base_model_s3_path,
)

xgb_train.set_hyperparameters(
    objective='binary:logistic',
    num_round=100,
    max_depth=4,
    eta=0.1,
    gamma=4,
    reg_lambda=20,
    min_child_weight=6,
    subsample=0.9,
    eval_metric='auc',
)

In [ ]:
xgb_train.fit(
    job_name=training_job_name,
    inputs={
        'train': TrainingInput(
            s3_data=preprocessed_train_data,
            content_type='text/csv',
        ),
        'validation': TrainingInput(
            s3_data=preprocessed_val_data,
            content_type='text/csv',
        ),
    },
)

## Deploy Model

Finally, we deploy the preprocessing model and trained model in one real-time endpoint using [Amazon SageMaker Inference Pipeline](https://docs.aws.amazon.com/sagemaker/latest/dg/inference-pipelines.html).

In [ ]:
processing_model_file = f'{processing_model_s3_path}/model.tar.gz'
processing_model_file

In [ ]:
sklearn_model = SKLearnModel(
   sagemaker_session=sagemaker_session,
   role=role,
   framework_version='1.2-1',
   model_data=processing_model_file,
   entry_point='preprocessor_inference.py', 
   source_dir='scripts/', 
)

xgboost_model = xgb_train.create_model(
    role=role,
    image_uri=image_uri,
    predictor_cls=XGBoostPredictor,
)

pipeline_model_name = f'responsible-model-{dt.datetime.now().strftime("%Y-%m-%d-%H-%M-%S")}'
pipeline_model = PipelineModel(
    sagemaker_session=sagemaker_session,
    role=role,
    models=[sklearn_model, xgboost_model],
    name=pipeline_model_name,
)
pipeline_model_name

In [ ]:
endpoint_name = f'responsible-model-endpoint-{dt.datetime.now().strftime("%Y-%m-%d-%H-%M-%S")}'
pipeline_model.deploy(
    endpoint_name=endpoint_name,
    instance_type=INSTANCE_TYPE, 
    initial_instance_count=1, 
)
endpoint_name

## Test predicting data

Let's evaluate the performance on the test data `test.csv`.

In [ ]:
predictor = Predictor(endpoint_name=endpoint_name)

In [ ]:
test_df = pd.read_csv(TEST_FILE)
test_labels = test_df[LABEL_COLUMN]
feature_df = test_df.drop(LABEL_COLUMN, axis=1, inplace=False)

payload = feature_df.iloc[:].to_csv(header=False, index=False)
p = predictor.predict(payload, initial_args={"ContentType": "text/csv"})
# print(p.decode("utf-8"))

In [ ]:
prediction = list(map(lambda x: float(x), p.decode("utf-8").split('\n')[:-1]))

In [ ]:
roc_auc_score(test_labels.to_numpy(), prediction)

## Task 5: Perform model explainability on the new trained model

We perform model explainability using Amazon SageMaker Clarify. 

### Task validation: 
- list top 8 most important features (with ',' as delimiter). The order follows the importance of the features. For example: `feature1,feature2,feature3,feature4,feature5,feature6,feature7,feature8`
- Submit the top_8_features and the clarify bias processing name following this format `top_8_features|clarify_job_name`. For example: `feature1,feature2,feature3,feature4,feature5,feature6,feature7,feature8|responsible-clarify-explainability-123`.

In [ ]:
clarify_processor = clarify.SageMakerClarifyProcessor(
    role=role,
    instance_count=1,
    instance_type=INSTANCE_TYPE,
    sagemaker_session=sagemaker_session,
)

In [ ]:
pipeline_model.name

In [ ]:
model_config = clarify.ModelConfig(
    model_name=pipeline_model.name,  # specify the inference pipeline model name
    instance_type=INSTANCE_TYPE,
    instance_count=1,
    accept_type='text/csv',
)

In [ ]:
raw_train_df = pd.read_csv(TRAIN_FILE)

# drop the target column and get the median value for each columns as the baseline
baseline = raw_train_df.drop([LABEL_COLUMN], axis=1).median().values.astype('int').tolist()

We want to run the Kernel SHAP analysis using 2500 number of samples, `mean_abs` for the global SHAP values aggregation method, and `use_logit`. 

In [ ]:
shap_config = clarify.SHAPConfig(
    baseline=[baseline],
    num_samples=2500,
    agg_method='mean_abs',
    use_logit=True,
)

In [ ]:
explainability_output_path = f's3://{BUCKET}/responsible-clarify-explainability'
explainability_data_config = clarify.DataConfig(
    s3_data_input_path=test_s3_path,
    s3_output_path=explainability_output_path,
    label=LABEL_COLUMN,
    headers=DATASET_COLUMNS,
    dataset_type='text/csv',
)

In [ ]:
explainability_job_name = f'clarify-explainability-{dt.datetime.now().strftime("%Y-%m-%d-%H-%M-%S")}'
clarify_processor.run_explainability(
    job_name=explainability_job_name,
    data_config=explainability_data_config,
    model_config=model_config,
    explainability_config=shap_config,
)
explainability_job_name

In [ ]:
!aws s3 cp --recursive $explainability_output_path ./responsible_explainability_report/